# Option 1: Build a lightweight LLM evaluation harness. 

You are given a model endpoint (you can use a dummy or mock enpoint – return a fixed or randomised string) and a JSONL test file in this format: 
- {"id": "q1", "input": "What is the leave policy?", "expected": "14 days annual leave"} 
- {"id": "q2", "input": "Who approves travel claims?", "expected": "Direct manager"} 

Build a CLI tool or small service that: 
- Runs each test case against the endpoint 
- Scores each response (justify scoring mechanisms, given that they can vary) 
- Outputs a structured summary (pass rate, failures with reasons, any anomalies) 
- Handles endpoint errors gracefully Requirements: • The tests should run successfully 
- At least one unit test • Clear error handling for malformed input and endpoint failures 
- README: what it does, how to run it, what you’d add with more time

In [10]:
import json
import sys
from pprint import pprint

In [11]:
# --- mock endpoint ---
def call_model(prompt):
    if "leave" in prompt.lower():
        return "14 days annual leave"
    if "travel" in prompt.lower():
        return "Direct manager"
    return "Unknown"

In [12]:
# --- simple scoring (exact match) ---
def score(pred, expected):
    return pred.strip().lower() == expected.strip().lower()

In [17]:
# --- main ---
def run(file_path):
    total, passed = 0, 0
    failures = []

    with open(file_path) as f:
        for line in f:
            try:
                row = json.loads(line)
                input = row["input"]
                expected = row["expected"]
                pred = call_model(input)
                ok = score(pred, expected)

                total += 1
                if ok:
                    passed += 1
                else:
                    failures.append({
                        "id": row["id"],
                        "input": input,
                        "expected": expected,
                        "prediction": pred
                    })

            except Exception as e:
                failures.append({"error": str(e)})

    print(f"Pass rate: {passed}/{total} = {passed/total:.2f}")
    print("Failures:")
    pprint(failures)

In [18]:
run('test.jsonl')

Pass rate: 4/10 = 0.40
Failures:
[{'expected': '15 days annual leave',
  'id': 'q3',
  'input': 'What is the leave policy?',
  'prediction': '14 days annual leave'},
 {'expected': 'HR manager',
  'id': 'q4',
  'input': 'Who approves travel claims?',
  'prediction': 'Direct manager'},
 {'expected': 'Formal attire',
  'id': 'q5',
  'input': 'What is the dress code?',
  'prediction': 'Unknown'},
 {'expected': 'Performance based bonus',
  'id': 'q6',
  'input': 'Tell me about bonuses',
  'prediction': 'Unknown'},
 {'expected': '14 days',
  'id': 'q7',
  'input': 'What is the leave policy?',
  'prediction': '14 days annual leave'},
 {'expected': '',
  'id': 'q9',
  'input': 'Random question',
  'prediction': 'Unknown'}]
